# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Task:** 4-class classification (Growth / Recovery / Stable / Decline) of a page's next 28 days, from its previous 56 days of search and engagement signals. **Primary metric:** macro F1 (each class counts equally); accuracy is secondary.

**Candidates:** histogram gradient boosting (handles missing GA4 natively, captures interactions between volume, trend and position) and logistic regression (a simple, readable comparison). **Baselines** (same data, same split, same metric): always the majority class; "last window's direction continues"; "last window's move reverses"; and a **recent-slope rule** (a depth-3 tree using only the last-7-day trend `r7`) — the demanding one, added after the w06 scan showed `r7` alone carries most of the signal; and a two-feature **momentum tree** (`r7` and `d_impr`), added after it scored higher than the single-feature rule on validation.

In [1]:
import os, sys, json, datetime as dt
REPO_DIR = "/content/FlyRank-ML-Internship"
if os.path.isdir("/content") and not os.environ.get("CAPSTONE_LOCAL"):
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone https://github.com/joannadulay/FlyRank-ML-Internship.git {REPO_DIR}")
    os.chdir(REPO_DIR)
    os.system("pip install -q datasets duckdb scikit-learn huggingface_hub pyarrow")
sys.path.insert(0, "work/src")
import numpy as np, pandas as pd
import capstone_lib as L
os.makedirs(L.OUT_DIR, exist_ok=True)
plan, frames, TOKEN, LOCAL = L.start()
train, val = frames["train"], frames["val"]          # the sealed test frame is not touched in this notebook
print("cutoffs  train:", [str(t) for t in plan["train"]], "| val:", plan["val"])
print("rows     train:", len(train), "| val:", len(val))

cutoffs  train: ['2026-01-11', '2026-02-08', '2026-03-08', '2026-04-05'] | val: 2026-05-03
rows     train: 237571 | val: 85997


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
w = plan["window_days"]
rows = []
for name, Ts in [("train", plan["train"]), ("val", [plan["val"]]), ("test (sealed)", [plan["test"]])]:
    for T in Ts:
        rows.append(dict(split=name, cutoff=str(T), features_from=str(T - dt.timedelta(days=2 * w - 1)),
                         label_window_ends=str(T + dt.timedelta(days=w))))
print(pd.DataFrame(rows).to_string(index=False))
assert max(plan["train"]) + dt.timedelta(days=w) <= plan["val"], "a training label window reaches into validation"
assert plan["val"] + dt.timedelta(days=w) <= plan["test"], "validation labels reach into the test cutoff"
print("\nOK: no label window overlaps a later split's cutoff.")

        split     cutoff features_from label_window_ends
        train 2026-01-11    2025-11-17        2026-02-08
        train 2026-02-08    2025-12-15        2026-03-08
        train 2026-03-08    2026-01-12        2026-04-05
        train 2026-04-05    2026-02-09        2026-05-03
          val 2026-05-03    2026-03-09        2026-05-31
test (sealed) 2026-06-02    2026-04-08        2026-06-30

OK: no label window overlaps a later split's cutoff.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
F = L.FEATURES
fitted, preds, table = {}, {}, {}
for n, m in L.make_models().items():
    fitted[n], preds[n] = L.fit_predict(m, train, val)
    table[n] = L.score(val.y, preds[n])
base_fns = dict(majority=L.baseline_majority(train.y), persistence=L.baseline_persistence(), mean_reversion=L.baseline_mean_reversion(),
                recent_slope=L.baseline_recent_slope(train), momentum_tree=L.baseline_momentum_tree(train))
for n, fn in base_fns.items():
    preds["baseline:" + n] = fn(val)
    table["baseline:" + n] = L.score(val.y, preds["baseline:" + n])

summary = pd.DataFrame({k: {"macro_f1": v["macro_f1"], "accuracy": v["accuracy"], **{f"f1_{c}": v["per_class_f1"][c] for c in L.CLASSES}}
                        for k, v in table.items()}).T.round(3)
print(summary.to_string())

best_model = max(fitted, key=lambda k: table[k]["macro_f1"])
best_base = max(base_fns, key=lambda k: table["baseline:" + k]["macro_f1"])
lo, hi = L.cluster_bootstrap_diff(val.y.values, preds[best_model], preds["baseline:" + best_base], val.client.values)
d = table[best_model]["macro_f1"] - table["baseline:" + best_base]["macro_f1"]
print(f"\nChosen on validation: model = {best_model}, strongest baseline = {best_base}")
print(f"Macro F1 difference {d:+.3f}  (95% interval, resampling clients: {lo:+.3f} to {hi:+.3f})")
json.dump(dict(model=best_model, baseline=best_base), open("work/outputs/model_choice.json", "w"))

                         macro_f1  accuracy  f1_Growth  f1_Recovery  f1_Stable  f1_Decline
hist_gradient_boosting      0.437     0.527      0.397        0.321      0.335       0.695
logistic_regression         0.430     0.514      0.372        0.284      0.391       0.674
baseline:majority           0.134     0.365      0.000        0.000      0.535       0.000
baseline:persistence        0.300     0.427      0.266        0.000      0.420       0.515
baseline:mean_reversion     0.230     0.304      0.000        0.214      0.420       0.285
baseline:recent_slope       0.399     0.594      0.333        0.000      0.554       0.709
baseline:momentum_tree      0.450     0.591      0.375        0.245      0.459       0.720

Chosen on validation: model = hist_gradient_boosting, strongest baseline = momentum_tree
Macro F1 difference -0.013  (95% interval, resampling clients: -0.033 to +0.007)


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
from sklearn.inspection import permutation_importance
conf = np.array(table[best_model]["confusion"], dtype=float)
conf_share = pd.DataFrame(conf / conf.sum(1, keepdims=True).clip(1), index=[f"observed {c}" for c in L.CLASSES],
                          columns=[f"predicted {c}" for c in L.CLASSES]).round(2)
print(conf_share.to_string(), "\n")

v = val.assign(pred=preds[best_model])
v["vol_band"] = pd.qcut(v.rate_B, 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
v["ok"] = v.pred == v.y
print("Accuracy by volume band:\n", v.groupby("vol_band", observed=True).ok.mean().round(3).to_string(), "\n")
v["ga4"] = np.where(v.ga4_cov_B > 0, "GA4 reporting", "no GA4")
print("Accuracy by GA4 coverage:\n", v.groupby("ga4").ok.mean().round(3).to_string(), "\n")

samp = val.sample(min(20000, len(val)), random_state=L.CFG["seed"])
pi = permutation_importance(fitted[best_model], samp[F], samp.y, scoring="f1_macro", n_repeats=3, random_state=L.CFG["seed"])
imp = pd.Series(pi.importances_mean, index=F).sort_values(ascending=False)
print("Permutation importance (drop in macro F1 when the feature is shuffled):\n", imp.round(4).to_string())

reading = (f"During development (validation cutoff {plan['val']}) the {best_model.replace('_', ' ')} scored macro F1 "
           f"{table[best_model]['macro_f1']:.3f} against {table['baseline:' + best_base]['macro_f1']:.3f} for the strongest simple baseline "
           f"({best_base.replace('_', ' ')}); 95% interval for the difference {lo:+.3f} to {hi:+.3f}.")
json.dump(dict(features=F, reading=reading, val_scores=table, importance=imp.round(5).to_dict()),
          open("work/outputs/w05_dev_metrics.json", "w"), indent=1)
print("\n" + reading)

                   predicted Growth  predicted Recovery  predicted Stable  predicted Decline
observed Growth                0.44                0.00              0.25               0.31
observed Recovery              0.00                0.79              0.02               0.19
observed Stable                0.14                0.22              0.23               0.40
observed Decline               0.05                0.14              0.07               0.74 

Accuracy by volume band:
 vol_band
Q1 low     0.415
Q2         0.521
Q3         0.573
Q4 high    0.598 

Accuracy by GA4 coverage:
 ga4
GA4 reporting    0.539
no GA4           0.505 

Permutation importance (drop in macro F1 when the feature is shuffled):
 r7                0.1464
d_impr            0.0965
log_rate_B        0.0261
cv_B              0.0224
ctr_B             0.0196
d_pos             0.0089
avail_share_B     0.0024
pos_B             0.0020
eng_per_day_B     0.0013
d_ctr             0.0002
active_share_B    0.0000
o

In [5]:
v = val.assign(pred=preds[best_model])
big_fall = v.d_impr < -2          # roughly an 85%+ fall in one window
print("pages with a very deep fall:", int(big_fall.sum()), "| their labels:", v[big_fall].y.value_counts(normalize=True).round(2).to_dict())
sub = v[~big_fall]
print("model macro F1 all / without them:", round(L.score(v.y, v.pred)["macro_f1"], 3), "/", round(L.score(sub.y, sub.pred)["macro_f1"], 3))
print(f"strongest baseline ({best_base}) without them:", round(L.score(sub.y, preds["baseline:" + best_base][~big_fall.values])["macro_f1"], 3))
print("Recovery F1 without them:", round(L.score(sub.y, sub.pred)["per_class_f1"]["Recovery"], 3))

pages with a very deep fall: 494 | their labels: {'Decline': 0.54, 'Stable': 0.28, 'Recovery': 0.18}
model macro F1 all / without them: 0.437 / 0.437
strongest baseline (momentum_tree) without them: 0.449
Recovery F1 without them: 0.319


In [6]:
allf = pd.concat([train, val], ignore_index=True)
cuts = sorted(allf.cutoff.unique())
rows = []
for i in range(2, len(cuts)):                      # need at least 2 earlier months to train on
    tr, te = allf[allf.cutoff.isin(cuts[:i])], allf[allf.cutoff == cuts[i]]
    _, p = L.fit_predict(L.make_models()[best_model], tr, te)
    rows.append(dict(test_month=str(cuts[i].date()), n=len(te),
                     model=L.score(te.y, p)["macro_f1"],
                     persistence=L.score(te.y, L.baseline_persistence()(te))["macro_f1"],
                     recent_slope=L.score(te.y, L.baseline_recent_slope(tr)(te))["macro_f1"],
                     momentum_tree=L.score(te.y, L.baseline_momentum_tree(tr)(te))["macro_f1"],
                     majority=L.score(te.y, L.baseline_majority(tr.y)(te))["macro_f1"]))
bt = pd.DataFrame(rows)
bt["best_baseline"] = bt[["persistence", "recent_slope", "momentum_tree", "majority"]].max(axis=1)
bt["gain_vs_best_baseline"] = bt.model - bt.best_baseline
print(bt.round(3).to_string(index=False))
bt.round(4).to_json("work/outputs/w05_backtest.json", orient="records", indent=1)

test_month     n  model  persistence  recent_slope  momentum_tree  majority  best_baseline  gain_vs_best_baseline
2026-03-08 60233  0.449        0.253         0.297          0.446     0.102          0.446                  0.003
2026-04-05 77105  0.446        0.261         0.348          0.406     0.129          0.406                  0.039
2026-05-03 85997  0.437        0.300         0.399          0.450     0.134          0.450                 -0.013


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.